In [4]:
import findspark
from pyspark.sql import SparkSession
from pyspark import SparkContext
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline
from pyspark.sql import functions as F
from pyspark.ml.feature import RobustScaler, VectorAssembler
from pyspark.sql.functions import udf
from pyspark.ml.linalg import VectorUDT
from pyspark.sql.types import DoubleType

In [5]:
findspark.init("/usr/local/spark")

if SparkContext._active_spark_context is not None:
    sc = SparkContext.getOrCreate()
    sc.stop()

In [ ]:

spark = SparkSession.builder \
    .appName("BigDataLab") \
    .config("spark.eventLog.enabled", "true") \
    .config("spark.ui.showConsoleProgress", "true") \
    .master("spark://localhost:7077") \
    .getOrCreate()
    
spark.sparkContext.setLogLevel("ERROR")

In [7]:
df = spark.read.csv("hdfs://localhost:9000/user/hadoop/fraud.csv", header=True, inferSchema=True)

In [8]:
df.show(5)

+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|    type|  amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|   1| PAYMENT| 9839.64|C1231006815|     170136.0|     160296.36|M1979787155|           0.0|           0.0|      0|             0|
|   1| PAYMENT| 1864.28|C1666544295|      21249.0|      19384.72|M2044282225|           0.0|           0.0|      0|             0|
|   1|TRANSFER|   181.0|C1305486145|        181.0|           0.0| C553264065|           0.0|           0.0|      1|             0|
|   1|CASH_OUT|   181.0| C840083671|        181.0|           0.0|  C38997010|       21182.0|           0.0|      1|             0|
|   1| PAYMENT|11668.14|C2048537720|      41554.0|      29885.86|M1230701703|      

Удаляем ненужные колонки

In [9]:
df = df.drop('nameOrig', 'nameDest')

One-hot encoding колонки type

In [10]:
categories = df.select("type").distinct().collect()
categories = [row["type"] for row in categories]

for category in categories:
    df = df.withColumn(f"type_{category}", 
                        F.when(F.col("type") == category, 1).otherwise(0))
    
df = df.drop('type')
    
df.show(5)

+----+--------+-------------+--------------+--------------+--------------+-------+--------------+-------------+------------+-------------+------------+----------+
|step|  amount|oldbalanceOrg|newbalanceOrig|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|type_TRANSFER|type_CASH_IN|type_CASH_OUT|type_PAYMENT|type_DEBIT|
+----+--------+-------------+--------------+--------------+--------------+-------+--------------+-------------+------------+-------------+------------+----------+
|   1| 9839.64|     170136.0|     160296.36|           0.0|           0.0|      0|             0|            0|           0|            0|           1|         0|
|   1| 1864.28|      21249.0|      19384.72|           0.0|           0.0|      0|             0|            0|           0|            0|           1|         0|
|   1|   181.0|        181.0|           0.0|           0.0|           0.0|      1|             0|            1|           0|            0|           0|         0|
|   1|   181.0|       

Сделаем нормализацию колонок

In [11]:
def vector_to_double(v):
    return float(v[0])

vector_to_double_udf = udf(vector_to_double, DoubleType())

In [12]:
columns_norm = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']

scaled_df = df
for col_name in columns_norm:
    assembler = VectorAssembler(inputCols=[col_name], outputCol=f"{col_name}_vector")
    scaled_df = assembler.transform(scaled_df)
    
    scaler = RobustScaler(inputCol=f"{col_name}_vector", outputCol=f"{col_name}_scaled")
    scaler_worker = scaler.fit(scaled_df)
    scaled_df = scaler_worker.transform(scaled_df)
    
    scaled_df = scaled_df.withColumn(f"{col_name}_scaled", vector_to_double_udf(scaled_df[f"{col_name}_scaled"]))
    
    scaled_df = scaled_df.drop(f"{col_name}_vector")
    scaled_df = scaled_df.drop(col_name)

In [13]:
scaled_df.show()

+----+-------+--------------+-------------+------------+-------------+------------+----------+--------------------+--------------------+---------------------+---------------------+---------------------+
|step|isFraud|isFlaggedFraud|type_TRANSFER|type_CASH_IN|type_CASH_OUT|type_PAYMENT|type_DEBIT|       amount_scaled|oldbalanceOrg_scaled|newbalanceOrig_scaled|oldbalanceDest_scaled|newbalanceDest_scaled|
+----+-------+--------------+-------------+------------+-------------+------------+----------+--------------------+--------------------+---------------------+---------------------+---------------------+
|   1|      0|             0|            0|           0|            0|           1|         0| 0.05046637975001405|  1.5866159355416294|   1.1141356793685189|                  0.0|                  0.0|
|   1|      0|             0|            0|           0|            0|           1|         0|0.009561677301238277| 0.19815913160250673|  0.13473299198165523|                  0.0|        

In [14]:
scaled_df.describe().show()

25/03/28 18:07:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+--------------------+------------------+--------------------+---------------------+---------------------+---------------------+
|summary|              step|             isFraud|      isFlaggedFraud|      type_TRANSFER|       type_CASH_IN|      type_CASH_OUT|       type_PAYMENT|          type_DEBIT|     amount_scaled|oldbalanceOrg_scaled|newbalanceOrig_scaled|oldbalanceDest_scaled|newbalanceDest_scaled|
+-------+------------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+--------------------+------------------+--------------------+---------------------+---------------------+---------------------+
|  count|           6362620|             6362620|             6362620|            6362620|            6362620|            6362620|            6362620|             636